In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import pint
import pint_xarray

In [ ]:
seapopym_v1 = xr.open_zarr(
    "/Users/adm-lehodey/Documents/Workspace/Projects/phd_optimization/notebooks/Article_1/data/2_global_simulation/biomass_global.zarr"
)
seapopym_v1 = (
    seapopym_v1["biomass"]
    .squeeze()
    .rename({"T": "time", "X": "x", "Y": "y"})
    .sel(time=slice("2000", "2019"))
    .load()
)
seapopym_v1.x.attrs = {}
seapopym_v1.y.attrs = {}
seapopym_v1 = seapopym_v1.pint.quantify().pint.to("g/m^2").pint.dequantify()
seapopym_v1

In [ ]:
seapopym_v2 = xr.open_zarr(
    "/Users/adm-lehodey/Documents/Workspace/Projects/seapopym-message/notebook/zooplankton_no_transport_seapopym_v2.zarr"
)
seapopym_v2 = seapopym_v2.sel(time=slice("2000", "2019")).load()["biomass"]
seapopym_v2

In [ ]:
# Normaliser les timestamps pour l'alignement
# Option 1: Arrondir les deux à la date (floor à minuit)
seapopym_v1_normalized = seapopym_v1.assign_coords(time=seapopym_v1.time.dt.floor("D"))
seapopym_v2_normalized = seapopym_v2.assign_coords(time=seapopym_v2.time.dt.floor("D"))

# Vérifier l'alignement
print(
    f"v1 time range: {seapopym_v1_normalized.time.values[0]} to {seapopym_v1_normalized.time.values[-1]}"
)
print(
    f"v2 time range: {seapopym_v2_normalized.time.values[0]} to {seapopym_v2_normalized.time.values[-1]}"
)
print(f"v1 shape: {seapopym_v1_normalized.shape}")
print(f"v2 shape: {seapopym_v2_normalized.shape}")

# Calculer la différence
mape = (seapopym_v1_normalized - seapopym_v2_normalized) / seapopym_v1_normalized
mape

In [ ]:
mape.mean("time").plot()

In [ ]:
seapodym = xr.open_zarr(
    "/Users/adm-lehodey/Documents/Workspace/Projects/phd_optimization/notebooks/Article_1/data/1_global/post_processed_light_global_multiyear_bgc_001_033.zarr/"
)
seapodym = (
    seapodym["zooplankton"]
    .squeeze()
    .rename({"T": "time", "X": "x", "Y": "y"})
    .sel(time=slice("2000", "2019"))
    .load()
)
seapodym.x.attrs = {}
seapodym.y.attrs = {}
seapodym.attrs["units"] = "g/m^2"
seapodym = seapodym.assign_coords(time=seapodym.time.dt.floor("D"))
seapodym


In [ ]:
# Calculer la différence
mape_original = (seapodym - seapopym_v2_normalized) / seapodym
mape_original


In [ ]:
mape_original.mean("time").plot(vmax=0.5)